# Preprocessing support tiketa

U ovom notebooku demonstriramo kompletnu **pipeline za obradu teksta** support tiketa pre treniranja klasifikatora.

**Koraci:**
1. Učitavanje normalizovanih podataka iz keša (`data/raw/tickets.csv`)
2. Čišćenje teksta (`clean_text`)
3. Enkodiranje labela (`encode_labels`)
4. Podela na train/validation/test (`split_data`)
5. Tokenizacija (`tokenize`)
6. Kreiranje PyTorch Dataset objekta (`create_dataset`)

Podaci dolaze iz [`Tobi-Bueck/customer-support-tickets`](https://huggingface.co/datasets/Tobi-Bueck/customer-support-tickets) i već su normalizovani u [`src/data_loader.py`](../src/data_loader.py) u kolone `text` (subject + body) i `category` (queue).

## 1. Priprema okruženja

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.preprocessing import TextPreprocessor, load_dataframe

preprocessor = TextPreprocessor()
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Učitavanje podataka

Učitavamo keširani CSV fajl koji sadrži normalizovane kolone `text` i `category`.

In [ ]:
df = load_dataframe()
print(f"Broj tiketa: {len(df):,}")
print(f"Broj klasa: {df[config.LABEL_COLUMN].nunique()}")
print(f"Tokenizer: {config.TOKENIZER_NAME}")
df.head()

## 3. Čišćenje teksta — `clean_text()`

Metoda `clean_text()` uklanja HTML tagove, URL-ove i specijalne karaktere, pretvara tekst u mala slova i normalizuje razmake.

In [ ]:
sample_texts = df[config.TEXT_COLUMN].head(5).tolist()
demo_rows = []

for i, raw_text in enumerate(sample_texts, start=1):
    cleaned = preprocessor.clean_text(raw_text)
    demo_rows.append({
        "primer": i,
        "pre": raw_text[:200] + ("..." if len(raw_text) > 200 else ""),
        "posle": cleaned[:200] + ("..." if len(cleaned) > 200 else ""),
        "duzina_pre": len(raw_text),
        "duzina_posle": len(cleaned),
    })

comparison_df = pd.DataFrame(demo_rows)
comparison_df

In [ ]:
sample = df.head(500).copy()
sample["cleaned_text"] = sample[config.TEXT_COLUMN].apply(preprocessor.clean_text)
sample["raw_length"] = sample[config.TEXT_COLUMN].str.len()
sample["clean_length"] = sample["cleaned_text"].str.len()

avg_lengths = pd.Series({
    "Pre ciscenja": sample["raw_length"].mean(),
    "Posle ciscenja": sample["clean_length"].mean(),
})

plt.figure(figsize=(8, 5))
sns.barplot(x=avg_lengths.index, y=avg_lengths.values, hue=avg_lengths.index, palette="pastel", legend=False)
plt.title("Prosecna duzina teksta pre i posle ciscenja (uzorak 500)")
plt.ylabel("Broj karaktera")
plt.tight_layout()
plt.show()

## 4. Enkodiranje labela — `encode_labels()`

Kategorije (queue departmani) se mapiraju u numeričke labele pomoću `config.LABEL2ID`.

In [ ]:
label_map = pd.DataFrame([
    {"category": label, "label_id": idx}
    for label, idx in config.LABEL2ID.items()
]).sort_values("label_id")
label_map

In [ ]:
example_label = df[config.LABEL_COLUMN].iloc[0]
encoded = preprocessor.encode_labels([example_label])[0]
print(f"Kategorija: {example_label}")
print(f"Enkodirana labela: {encoded}")

## 5. Podela podataka — `split_data()`

Stratifikovana podela na train (70%), validation (15%) i test (15%) uz fiksiran seed za reproduktivnost.

In [ ]:
train_df, val_df, test_df = preprocessor.split_data(df)

print(f"Train: {len(train_df):,} ({len(train_df) / len(df):.1%})")
print(f"Validation: {len(val_df):,} ({len(val_df) / len(df):.1%})")
print(f"Test: {len(test_df):,} ({len(test_df) / len(df):.1%})")
print(f"\nSacuvano u: {config.PROCESSED_DATA_DIR}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
splits = [("Train", train_df), ("Validation", val_df), ("Test", test_df)]

for ax, (name, split_df) in zip(axes, splits):
  counts = split_df[config.LABEL_COLUMN].value_counts()
  sns.barplot(x=counts.index, y=counts.values, ax=ax, color="steelblue")
  ax.set_title(f"{name} ({len(split_df):,})")
  ax.set_xlabel("Kategorija")
  ax.set_ylabel("Broj tiketa")
  ax.tick_params(axis="x", rotation=45)
  for label in ax.get_xticklabels():
    label.set_horizontalalignment("right")

plt.suptitle("Distribucija klasa po splitovima (stratifikacija)")
plt.tight_layout()
plt.show()

## 6. Tokenizacija — `tokenize()`

Koristimo HuggingFace `bert-base-uncased` tokenizer sa maksimalnom dužinom od 128 tokena.

In [ ]:
example_text = preprocessor.clean_text(df[config.TEXT_COLUMN].iloc[0])
encoding = preprocessor.tokenize(example_text, return_tensors=None)
tokens = preprocessor.tokenizer.convert_ids_to_tokens(encoding["input_ids"])

print(f"Broj tokena: {len(tokens)}")
print(f"Tokeni: {tokens[:20]}...")

In [ ]:
token_sample = train_df.head(500).copy()
token_sample["cleaned_text"] = token_sample[config.TEXT_COLUMN].apply(preprocessor.clean_text)
token_lengths = [
    len(preprocessor.tokenize(text, return_tensors=None)["input_ids"])
    for text in token_sample["cleaned_text"]
]

plt.figure(figsize=(10, 5))
sns.histplot(token_lengths, bins=30, kde=True, color="teal")
plt.axvline(config.MAX_SEQUENCE_LENGTH, color="red", linestyle="--", label=f"max_length={config.MAX_SEQUENCE_LENGTH}")
plt.title("Distribucija broja tokena (uzorak 500)")
plt.xlabel("Broj tokena")
plt.ylabel("Frekvencija")
plt.legend()
plt.tight_layout()
plt.show()

## 7. PyTorch Dataset — `create_dataset()`

Kreiramo `TicketDataset` koji vraća `input_ids`, `attention_mask` i `labels` za svaki tiket.

In [ ]:
train_dataset = preprocessor.create_dataset(train_df.head(32))
sample_item = train_dataset[0]

print(f"Velicina dataseta: {len(train_dataset)}")
print(f"input_ids shape: {sample_item['input_ids'].shape}")
print(f"attention_mask shape: {sample_item['attention_mask'].shape}")
print(f"label: {sample_item['labels'].item()} -> {config.ID2LABEL[sample_item['labels'].item()]}")

## Zaključak

Preprocessing pipeline je spreman:
- Tekstovi su očišćeni i normalizovani
- Labele su enkodirane u numeričke ID-jeve
- Podaci su podeljeni stratifikovano (70/15/15)
- Tokenizacija koristi `bert-base-uncased`
- PyTorch `TicketDataset` je spreman za treniranje

Sledeći korak: fine-tuning BERT modela na `train_dataset`.